<a href="https://colab.research.google.com/github/ubsuny/PHY386/blob/Homework2026/2026/FinalProject/Final_04_ExoplanetKOI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Project — Topic 4: Exoplanet Confirmation Classifier (42 pts)

## Learning Outcomes
- Query the **NASA Exoplanet Archive** through `astroquery` — the same library you used in HW6 for SDSS.
- Handle missing values in real astronomical catalog data.
- Train Decision Tree, Random Forest, and Multi-Layer Perceptron classifiers to separate CONFIRMED planets from FALSE POSITIVE detections.
- Use feature importances to connect the machine-learning result back to transit-photometry physics.
- Apply the best model to the CANDIDATE list and produce a ranked table of most-likely-real planets.

## GitHub Workflow
Fork → `yourname-final` → PR into `Homework2026`. Reviewer `@laserlab`, milestone `Final-2026`.

## Background: Transits and False Positives

The **Kepler** space telescope (2009–2018) monitored ~150,000 stars in a single patch of sky, looking for **transits**: a planet passing in front of its host star dims the star by a tiny fraction for some time, periodically. The depth of the dip is $\delta \approx (R_p / R_\star)^2$ — about 1% for a Jupiter, 0.01% for an Earth.

The problem: many *other* things also produce periodic dips.
- **Eclipsing binary stars** — two stars orbit each other, eclipse periodically. Dips are much deeper (often >>10%) and the transit shape is V-rather-than-U.
- **Background eclipsing binaries** — a faint binary system aligned with your target star dilutes into a small transit-like signal.
- **Instrumental artifacts** — cosmic rays, pixel-sensitivity changes, spacecraft motion.

The **Kepler Objects of Interest (KOI)** catalog tracks every candidate Kepler ever found, with three labels:
- `CONFIRMED` — validated by follow-up observations or statistical arguments.
- `CANDIDATE` — still being vetted.
- `FALSE POSITIVE` — ruled out.

Can you distinguish CONFIRMED from FALSE POSITIVE using only the transit parameters the pipeline measures? And if so, can you apply the model to rank the remaining CANDIDATES?

In [ ]:
# Required packages (uncomment for Colab or fresh environment):
# %pip install astroquery scikit-learn pandas --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.size': 11, 'axes.labelsize': 11, 'axes.titlesize': 11,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
})

## Worked Example: Pull the KOI Cumulative Table

We use `astroquery.ipac.nexsci.nasa_exoplanet_archive.NasaExoplanetArchive.query_criteria()`. The target table is `cumulative` — the Kepler Objects of Interest cumulative table. The `select=` string lists the columns. Full column catalog: [NASA Exoplanet Archive cumulative table](https://exoplanetarchive.ipac.caltech.edu/docs/API_kepcandidate_columns.html).

In [ ]:
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

tab = NasaExoplanetArchive.query_criteria(
    table="cumulative",
    select=(
        "kepid, kepoi_name, koi_disposition, "
        "koi_period, koi_duration, koi_depth, koi_prad, "
        "koi_steff, koi_slogg, koi_srad, koi_impact, koi_model_snr"
    ),
)
df = tab.to_pandas()
print(f"{len(df)} objects of interest")
print(df['koi_disposition'].value_counts())
df.head()

**Feature dictionary** (all come from the transit-fitting pipeline):

| Column | Meaning |
|--------|---------|
| `koi_period` | orbital period (days) |
| `koi_duration` | transit duration (hours) |
| `koi_depth` | transit depth (parts per million) |
| `koi_prad` | inferred planet radius (Earth radii) |
| `koi_steff` | host-star effective temperature (K) |
| `koi_slogg` | host-star log-g (cgs) |
| `koi_srad` | host-star radius (solar radii) |
| `koi_impact` | impact parameter (0 = central transit, 1 = grazing) |
| `koi_model_snr` | pipeline signal-to-noise of the transit fit |

## Part 1 — Data Cleaning (8 pts)

### Task 1.1 — Drop or impute missing values (4 pts)
Count missing values per column. Choose a strategy (drop-rows vs. median-impute) and justify it. Record how many rows remain.

### Task 1.2 — Separate CONFIRMED/FALSE POSITIVE from CANDIDATE (4 pts)
Split the DataFrame into two:
- `labeled` — rows with `koi_disposition` in `{CONFIRMED, FALSE POSITIVE}`. This is your training+test set, a **binary classification problem** (label = 1 if CONFIRMED, 0 if FALSE POSITIVE).
- `candidates` — rows labeled `CANDIDATE`. You'll apply the trained model to these later.

Check class balance in `labeled`.

In [ ]:
# Part 1: your code here

## Part 2 — Train Three Classifiers (14 pts)

### Task 2.1 — Train/test split and scaling (2 pts)
80/20 split with `random_state=42`, stratified on the label. Standardize features with `StandardScaler` (fit on train only).

### Task 2.2 — Decision Tree (3 pts)
Train a `DecisionTreeClassifier` with `max_depth=5`. Report accuracy, precision, recall, and the confusion matrix.

### Task 2.3 — Random Forest (4 pts)
Train a `RandomForestClassifier` with 200 trees. Same metrics. Also plot `feature_importances_` as a horizontal bar chart.

### Task 2.4 — Multi-Layer Perceptron (5 pts)
Train sklearn's `MLPClassifier` with two hidden layers of 32 units, `max_iter=500`. Same metrics. Compare training curve (`loss_curve_`) across models.

In [ ]:
# Part 2: your code here
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.neural_network import MLPClassifier
# from sklearn.preprocessing import StandardScaler
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import classification_report, confusion_matrix

## Part 3 — Physics Interpretation of Feature Importances (8 pts)

### Task 3.1 — Rank and explain (4 pts)
Look at Random Forest's top-3 features. For each, explain in 1–2 sentences *why* that feature would physically help distinguish a real planet from a false positive. Hints:
- `koi_depth`: eclipsing binaries produce much deeper dips than planets.
- `koi_prad`: inferred radii above ~2 R_Jupiter almost have to be stars.
- `koi_model_snr`: low-SNR detections are more prone to instrumental artifacts.

### Task 3.2 — Diagnostic scatter plots (4 pts)
Plot `koi_depth` vs. `koi_prad` (log–log axes), colored by class. Repeat for `koi_duration` vs. `koi_period`. Circle any region where the classes overlap strongly and comment.

In [ ]:
# Part 3: your code here

## Part 4 — Rank the CANDIDATE list (8 pts)

### Task 4.1 — Predict probabilities (4 pts)
Apply your best classifier's `predict_proba` to the `candidates` DataFrame. Sort by predicted "CONFIRMED" probability, descending. Print the top 20 and the bottom 20.

### Task 4.2 — Cross-check with recent confirmations (4 pts)
Look up 2–3 names from your top-20 list in the [NASA Exoplanet Archive web UI](https://exoplanetarchive.ipac.caltech.edu/) or in the literature. Have any been confirmed *after* the snapshot you downloaded? Did your model "predict the future" for any?

In [ ]:
# Part 4: your code here

## Part 5 — Stretch: Cross-Mission Generalization (4 pts, optional bonus)

Re-run the query with `table="toi"` (TESS Objects of Interest) and a similar column list (column names differ — see the [TOI docs](https://exoplanetarchive.ipac.caltech.edu/docs/API_toi_columns.html)). Does your Kepler-trained classifier generalize to TESS, or does it need retraining? Why?

In [ ]:
# Part 5 (optional): your code here

## Submission
**Good commit** ✅: `Add Random Forest classifier and feature-importance plot`

**Bad commit** ❌: `wip`

### Checklist
- [ ] Notebook in `2026/FinalProject/<yourname>/`
- [ ] Runs top-to-bottom on a fresh kernel
- [ ] Type annotations and NumPy docstrings on your functions
- [ ] All plots labeled with units
- [ ] Tasks sum to ~42 points
- [ ] PR into `Homework2026`, reviewer `@laserlab`, milestone `Final-2026`
- [ ] AI usage disclosed

## Resources
- [`astroquery` Exoplanet Archive docs](https://astroquery.readthedocs.io/en/latest/ipac/nexsci/nasa_exoplanet_archive.html)
- [NASA Exoplanet Archive KOI column dictionary](https://exoplanetarchive.ipac.caltech.edu/docs/API_kepcandidate_columns.html)
- Thompson, S. E. et al. (2018). *Kepler Data Release 25 DR25 KOI Table.* ApJS 235, 38.
- Shallue & Vanderburg (2018), *Identifying Exoplanets with Deep Learning,* AJ 155, 94. — neural nets on Kepler light curves.